# Day 4: SQL Window Functions — Interactive Practice

## Objective
Learn how to use SQL window functions to perform calculations across rows without collapsing them. By the end of this notebook, you will be comfortable with `OVER()`, `PARTITION BY`, ranking functions, offset functions (`LAG`/`LEAD`), running totals, moving averages, and distribution functions.

## Table of Contents
1. [AVG() OVER() — Company-Wide Average on Every Row](#1-avg-over--company-wide-average-on-every-row)
2. [AVG() OVER(PARTITION BY) — Department Average, Rows Not Collapsed](#2-avg-overpartition-by--department-average-rows-not-collapsed)
3. [ROW_NUMBER, RANK, DENSE_RANK — Three Ranking Functions](#3-row_number-rank-dense_rank--three-ranking-functions)
4. [Top-N Per Group — Top 3 Earners Per Department](#4-top-n-per-group--top-3-earners-per-department)
5. [LAG — Previous Employee's Salary](#5-lag--previous-employees-salary)
6. [LEAD — Next Sale Date and Days Between Sales](#6-lead--next-sale-date-and-days-between-sales)
7. [Running Total — Cumulative Sales Amount Over Time](#7-running-total--cumulative-sales-amount-over-time)
8. [Moving Average — 3-Month Rolling Average](#8-moving-average--3-month-rolling-average)
9. [PERCENT_RANK — Salary Percentile Within Department](#9-percent_rank--salary-percentile-within-department)
10. [NTILE(4) — Salary Quartiles Per Department](#10-ntile4--salary-quartiles-per-department)
11. [Complex Real-World Query — Multiple Window Functions + CASE WHEN](#11-complex-real-world-query--multiple-window-functions--case-when)
12. [Try It Yourself Exercises](#12-try-it-yourself-exercises)

## Connection Setup

We use the same `run_query` helper from previous days. It connects to the PostgreSQL database, executes a query, and returns results as a pandas DataFrame.

**Tables we will use:**
- `company.employees` — emp_id, first_name, last_name, email, department_id, salary, hire_date, manager_id, is_active
- `company.departments` — dept_id, dept_name, location, budget
- `company.sales` — sale_id, employee_id, product_id, quantity, sale_date, region
- `company.products` — product_id, product_name, category, unit_price

**Note on sales revenue:** The `sales` table stores `quantity` but not the monetary amount. To get revenue, we join with `products` and compute `quantity * unit_price`.

In [ ]:
import psycopg2
import pandas as pd

def run_query(sql: str) -> pd.DataFrame:
    """Run a SQL query and return results as a DataFrame."""
    conn = psycopg2.connect(
        host="localhost", port=5432,
        dbname="week2_db", user="student", password="student123"
    )
    try:
        df = pd.read_sql_query(sql, conn)
        return df
    finally:
        conn.close()

print("Helper function defined. Ready for window functions!")

### Quick Schema Reminder

Let's verify we can access the company schema and see the relevant tables.

In [ ]:
run_query("""
    SELECT table_name 
    FROM information_schema.tables 
    WHERE table_schema = 'company'
    ORDER BY table_name
""")

**Expected output:** Four tables — departments, employees, products, sales.

---

## 1. AVG() OVER() — Company-Wide Average on Every Row

A window function with an **empty** `OVER()` clause computes a value across the entire result set, but unlike `GROUP BY`, it does **not collapse rows**. Every row keeps its individual data, and the aggregate appears as an extra column on every row.

**Plain English:** "Show me every employee with their salary, and also the company-wide average salary on each row."

**Key concept:** `AVG(salary) OVER()` — the empty `OVER()` means the window includes all rows.

In [ ]:
run_query("""
    SELECT 
        first_name,
        last_name,
        department_id,
        salary,
        ROUND(AVG(salary) OVER(), 2) AS company_avg_salary
    FROM company.employees
    ORDER BY emp_id
""")

**Expected output:** Every employee appears with their own salary, and the `company_avg_salary` column shows the same value on every row (around 84,000). The average is computed across ALL employees, including inactive ones and the one without a department.

**Compare with GROUP BY:** If we had used `GROUP BY department_id`, we would get only 7 rows (one per department_id value). With `OVER()`, we get all 51 rows — no collapse!

---

## 2. AVG() OVER(PARTITION BY) — Department Average, Rows Not Collapsed

Adding `PARTITION BY department_id` tells PostgreSQL: "compute the average, but only within each department partition." The result is a department-average value on every employee row, and rows are still not collapsed.

**Plain English:** "Show me every employee, their salary, and the average salary of their department — all on the same row."

**This is the key difference from GROUP BY:** GROUP BY collapses to one row per department. `OVER(PARTITION BY)` keeps every individual row.

In [ ]:
run_query("""
    SELECT 
        e.first_name,
        e.last_name,
        d.dept_name,
        e.salary,
        ROUND(AVG(e.salary) OVER(PARTITION BY e.department_id), 2) AS dept_avg_salary,
        COUNT(*) OVER(PARTITION BY e.department_id) AS dept_employee_count
    FROM company.employees e
    LEFT JOIN company.departments d ON e.department_id = d.dept_id
    ORDER BY e.department_id, e.emp_id
""")

**Expected output:** Every employee on their own row. Employees in the same department share the same `dept_avg_salary` and `dept_employee_count`. The employee with `NULL` department_id (Paula Morris) appears alone with her own values.

**Key insight:** We used TWO window functions in the same query — `AVG()` and `COUNT()` — each with its own `PARTITION BY`. They operate independently on the same set of rows.

**Verify no collapse:** Count the rows — you should get all 51 employees. A `GROUP BY department_id` query would return only 7 rows (6 departments + NULL).

In [ ]:
# Verify: count of rows (should be 51)
run_query("""
    SELECT COUNT(*) AS total_rows
    FROM company.employees
""")

**Expected output:** `total_rows = 51`. This confirms that window functions preserve individual rows.

---

## 3. ROW_NUMBER, RANK, DENSE_RANK — Three Ranking Functions

These three functions rank rows within a partition, but they handle **ties** differently:

| Function | Behavior with ties | Example sequence |
|----------|-------------------|------------------|
| `ROW_NUMBER()` | Assigns unique sequential numbers, breaks ties arbitrarily | 1, 2, 3, 4, 5 |
| `RANK()` | Skips numbers after ties | 1, 2, 2, 4, 5 |
| `DENSE_RANK()` | Does NOT skip numbers after ties | 1, 2, 2, 3, 4 |

Our dataset has employees with the same salary within a department (e.g., David Kim and Eva Patel both earn 88,000 in Engineering; Quinn Walker and Rachel Hall both earn 78,000 in Finance). This makes the difference visible.

**Plain English:** "Rank employees by salary within each department using all three methods so we can see the difference."

In [ ]:
run_query("""
    SELECT 
        e.first_name || ' ' || e.last_name AS full_name,
        d.dept_name,
        e.salary,
        ROW_NUMBER() OVER(PARTITION BY e.department_id ORDER BY e.salary DESC) AS row_num,
        RANK()       OVER(PARTITION BY e.department_id ORDER BY e.salary DESC) AS rank_num,
        DENSE_RANK() OVER(PARTITION BY e.department_id ORDER BY e.salary DESC) AS dense_rank_num
    FROM company.employees e
    LEFT JOIN company.departments d ON e.department_id = d.dept_id
    ORDER BY e.department_id, e.salary DESC, e.emp_id
""")

**Expected output:** Within each department, you'll see all three ranking columns. Look for these key differences:

- **Engineering:** David Kim and Eva Patel both earn 88,000. `RANK()` gives them both rank 3, then Frank Wilson gets rank 5 (skipped 4). `DENSE_RANK()` gives them both rank 3, then Frank gets rank 4 (no skip). `ROW_NUMBER()` arbitrarily assigns one 3 and the other 4.
- **Finance:** Quinn Walker and Rachel Hall both earn 78,000. Same pattern — `RANK()` skips, `DENSE_RANK()` doesn't.
- **Sales:** Cindy Nelson and Derek Carter have different salaries but same hire_date — ranking by salary shows no tie here.

**When to use which:**
- `ROW_NUMBER()`: When you need a unique identifier per row (e.g., "pick the top 1")
- `RANK()`: When ties should share a rank and you want to know "how many are strictly better"
- `DENSE_RANK()`: When ties share a rank but you want consecutive numbering (e.g., "top 3 salary levels" regardless of how many people share each level)

---

## 4. Top-N Per Group — Top 3 Earners Per Department

A common real-world pattern: "For each department, show me the top 3 highest-paid employees." This combines `ROW_NUMBER()` (or `RANK()`) with a CTE and a filter.

**Why a CTE?** Window functions cannot appear in a `WHERE` clause — they are computed after `WHERE` filtering. So we compute the rank in a CTE (or subquery), then filter in the outer query.

**Plain English:** "Rank employees by salary within each department, then keep only the top 3."

In [ ]:
run_query("""
    WITH ranked AS (
        SELECT 
            e.first_name || ' ' || e.last_name AS full_name,
            d.dept_name,
            e.salary,
            ROW_NUMBER() OVER(PARTITION BY e.department_id ORDER BY e.salary DESC) AS rn
        FROM company.employees e
        LEFT JOIN company.departments d ON e.department_id = d.dept_id
    )
    SELECT 
        dept_name,
        full_name,
        salary,
        rn AS rank_in_dept
    FROM ranked
    WHERE rn <= 3
    ORDER BY dept_name, rn
""")

**Expected output:** Up to 3 employees per department (the employee with NULL department appears as their own "group"). Each department's highest earners are shown in order.

**What if we used RANK() instead?** If the 3rd and 4th employees have the same salary, `RANK()` would give them both rank 3, and both would pass the `WHERE rn <= 3` filter — you might get more than 3 rows for that department. `ROW_NUMBER()` guarantees exactly 3 (or fewer if the department has fewer employees).

**Challenge:** Try changing `ROW_NUMBER()` to `RANK()` and see if any department gets more than 3 rows.

---

## 5. LAG — Previous Employee's Salary

`LAG(column, offset)` lets you look at a **previous row's** value within the same partition. The default offset is 1 (the immediately preceding row).

**Plain English:** "For each employee in a department, ordered by hire_date, show the salary of the employee hired just before them, and calculate the difference."

**Use case:** Comparing how salaries change over time within a team, or detecting anomalies.

In [ ]:
run_query("""
    SELECT 
        e.first_name || ' ' || e.last_name AS full_name,
        d.dept_name,
        e.hire_date,
        e.salary,
        LAG(e.salary, 1) OVER(PARTITION BY e.department_id ORDER BY e.hire_date) AS prev_salary,
        e.salary - LAG(e.salary, 1) OVER(PARTITION BY e.department_id ORDER BY e.hire_date) AS salary_diff
    FROM company.employees e
    LEFT JOIN company.departments d ON e.department_id = d.dept_id
    ORDER BY e.department_id, e.hire_date, e.emp_id
""")

**Expected output:** Each employee appears in hire_date order within their department. The first employee in each department has `NULL` for `prev_salary` and `salary_diff` (there is no previous hire). Subsequent employees show the salary of whoever was hired before them in that department, and the difference.

**Notice:** David Kim and Eva Patel in Engineering have the same hire_date (2022-01-15) and the same salary (88,000). The ORDER BY hire_date orders them arbitrarily relative to each other — one will see the other's salary as `prev_salary`.

**LAG with larger offset:** `LAG(salary, 2)` would skip back two rows instead of one. `LAG(salary, 1, 0)` uses 0 as the default value instead of NULL for the first row.

---

## 6. LEAD — Next Sale Date and Days Between Sales

`LEAD(column, offset)` is the mirror of `LAG` — it looks **forward** to the next row instead of backward.

**Plain English:** "For each sale, show the date of the next sale (any product, any employee), and compute how many days passed between them."

**Use case:** Understanding sales velocity, gaps in activity, or customer purchase cycles.

In [ ]:
run_query("""
    SELECT 
        s.sale_id,
        s.sale_date,
        LEAD(s.sale_date) OVER(ORDER BY s.sale_date) AS next_sale_date,
        LEAD(s.sale_date) OVER(ORDER BY s.sale_date) - s.sale_date AS days_until_next_sale
    FROM company.sales s
    ORDER BY s.sale_date, s.sale_id
    LIMIT 30
""")

**Expected output:** Each sale with its date, the date of the next sale chronologically, and the gap in days. The last sale has `NULL` for `next_sale_date` and `days_until_next_sale` because there is no subsequent sale.

**PostgreSQL date arithmetic:** Subtracting two `DATE` values gives an `INTEGER` representing the number of days between them.

**Real-world extension:** You could partition by region to see days-between-sales within each region, or by employee to analyze individual salesperson activity patterns.

---

## 7. Running Total — Cumulative Sales Amount Over Time

A running (cumulative) total is one of the most common uses of window functions. We use `SUM() OVER(ORDER BY ...)` with the **default frame** (`RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`), which means "sum all rows from the start up to the current one."

**Plain English:** "Show each sale with its revenue, and a running total of all sales up to that point, partitioned by region."

**Key syntax:** `SUM(amount) OVER(PARTITION BY region ORDER BY sale_date)` — the `ORDER BY` inside `OVER()` activates the cumulative frame.

In [ ]:
run_query("""
    SELECT 
        s.sale_id,
        s.sale_date,
        s.region,
        s.quantity * p.unit_price AS sale_amount,
        ROUND(SUM(s.quantity * p.unit_price) OVER(
            PARTITION BY s.region 
            ORDER BY s.sale_date
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ), 2) AS running_total
    FROM company.sales s
    JOIN company.products p ON s.product_id = p.product_id
    WHERE s.region IS NOT NULL
    ORDER BY s.region, s.sale_date, s.sale_id
    LIMIT 40
""")

**Expected output:** For each region, sales are ordered by date. The `running_total` starts at the first sale's amount and grows with each subsequent sale. When the region changes, the running total resets to 0 and starts accumulating again.

**Frame clause explained:** `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` means "from the very first row in the partition up to and including this row." This is actually the default when you use `ORDER BY` inside `OVER()`, but writing it explicitly makes your intent clear.

**Note:** We excluded `NULL` region sales to keep the running totals clean. In practice, you might want to assign those to an "Unknown" region.

---

## 8. Moving Average — 3-Month Rolling Average

A moving (rolling) average smooths out fluctuations by averaging over a sliding window of recent values. Here, we compute monthly total sales, then average over a 3-month window.

**Two-step approach:**
1. CTE: Aggregate sales to monthly totals
2. Window: Compute a 3-month moving average

**Key syntax:** `AVG(monthly_total) OVER(ORDER BY sale_month ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)` — this averages the current month plus the 2 preceding months.

**Plain English:** "What is the rolling 3-month average of total monthly sales?"

In [ ]:
run_query("""
    WITH monthly_sales AS (
        SELECT 
            DATE_TRUNC('month', s.sale_date) AS sale_month,
            ROUND(SUM(s.quantity * p.unit_price), 2) AS monthly_total
        FROM company.sales s
        JOIN company.products p ON s.product_id = p.product_id
        GROUP BY DATE_TRUNC('month', s.sale_date)
    )
    SELECT 
        sale_month,
        monthly_total,
        ROUND(AVG(monthly_total) OVER(
            ORDER BY sale_month 
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ), 2) AS rolling_3month_avg
    FROM monthly_sales
    ORDER BY sale_month
""")

**Expected output:** 12 rows (one per month from January 2025 through December 2025). The first month's rolling average equals its own monthly total (only 1 month in the window). The second month averages months 1-2. From month 3 onward, it averages the current plus 2 preceding months.

**Why ROWS instead of RANGE:** `ROWS BETWEEN 2 PRECEDING AND CURRENT ROW` means "the 2 physical rows before this one." If we used `RANGE`, it would consider all rows with the same `sale_month` value (which works too since each month is unique, but `ROWS` is more explicit for this pattern).

**Business insight:** The rolling average smooths out monthly spikes. If December has a huge year-end push, the rolling average will show a more gradual increase compared to the raw monthly total.

---

## 9. PERCENT_RANK — Salary Percentile Within Department

`PERCENT_RANK()` returns a value between 0 and 1 indicating the relative position of a row within its partition. It is calculated as `(rank - 1) / (total_rows - 1)`.

- 0.0 = lowest value in the partition
- 1.0 = highest value in the partition
- 0.5 = median position

**Plain English:** "What percentile is each employee's salary within their department?"

**Use case:** Performance reviews, compensation analysis, identifying outliers.

In [ ]:
run_query("""
    SELECT 
        e.first_name || ' ' || e.last_name AS full_name,
        d.dept_name,
        e.salary,
        ROUND(PERCENT_RANK() OVER(
            PARTITION BY e.department_id 
            ORDER BY e.salary
        )::numeric, 3) AS salary_percentile
    FROM company.employees e
    LEFT JOIN company.departments d ON e.department_id = d.dept_id
    ORDER BY e.department_id, e.salary DESC
""")

**Expected output:** Within each department, the highest earner gets 1.0 (100th percentile), the lowest gets 0.0 (0th percentile). Employees in between get proportional values. Tied salaries get the same percentile rank.

**Example interpretation:** An employee with `salary_percentile = 0.750` earns more than 75% of their department colleagues (they are at the 75th percentile).

**The `::numeric` cast:** PostgreSQL's `PERCENT_RANK()` returns a `double precision` type. Casting to `numeric` and rounding gives cleaner output. Without the cast, you'd see many decimal places.

---

## 10. NTILE(4) — Salary Quartiles Per Department

`NTILE(n)` divides rows in a partition into `n` roughly equal groups, numbered 1 through n. `NTILE(4)` creates quartiles — each department's employees are split into 4 salary-based groups.

**Plain English:** "Divide employees in each department into 4 salary quartiles."

**Use case:** Compensation bands, identifying which quartile an employee falls into for budgeting.

In [ ]:
run_query("""
    SELECT 
        e.first_name || ' ' || e.last_name AS full_name,
        d.dept_name,
        e.salary,
        NTILE(4) OVER(PARTITION BY e.department_id ORDER BY e.salary DESC) AS salary_quartile
    FROM company.employees e
    LEFT JOIN company.departments d ON e.department_id = d.dept_id
    ORDER BY e.department_id, e.salary DESC
""")

**Expected output:** Within each department, employees are numbered 1-4 based on their salary rank. Quartile 1 = highest paid, Quartile 4 = lowest paid (because we ordered by `salary DESC`).

**Example:** Engineering has 8 employees. NTILE(4) assigns 2 people to each quartile. Sales has 9 employees, so one quartile gets an extra person (NTILE distributes remainders starting from the first group).

**NTILE vs PERCENT_RANK:** `NTILE(4)` gives discrete groups (1, 2, 3, 4). `PERCENT_RANK()` gives a continuous value between 0 and 1. Use NTILE when you want clear categories; use PERCENT_RANK for precise positioning.

---

## 11. Complex Real-World Query — Multiple Window Functions + CASE WHEN

Now let's combine everything into one realistic business report. Management wants a single query showing, for each employee:

- Their name and department
- Their salary
- Their department's average salary
- The company-wide average salary
- Their rank within the department (by salary)
- A label: "Above Average" or "Below Average" compared to their department average

This requires **four** window functions and a `CASE WHEN` expression — all in one query, no CTEs needed.

In [ ]:
run_query("""
    SELECT 
        e.first_name || ' ' || e.last_name AS full_name,
        d.dept_name,
        e.salary,
        ROUND(AVG(e.salary) OVER(PARTITION BY e.department_id), 2) AS dept_avg_salary,
        ROUND(AVG(e.salary) OVER(), 2) AS company_avg_salary,
        RANK() OVER(PARTITION BY e.department_id ORDER BY e.salary DESC) AS rank_in_dept,
        CASE 
            WHEN e.salary > AVG(e.salary) OVER(PARTITION BY e.department_id) 
                THEN 'Above Average'
            WHEN e.salary < AVG(e.salary) OVER(PARTITION BY e.department_id) 
                THEN 'Below Average'
            ELSE 'Exactly Average'
        END AS vs_department_avg
    FROM company.employees e
    LEFT JOIN company.departments d ON e.department_id = d.dept_id
    ORDER BY e.department_id, e.salary DESC
""")

**Expected output:** A rich analytical table. Each employee is shown with their salary, their department's average, the company average, their rank, and a human-readable label.

**Key observations:**
- Directors (Alice Chen, Iris Taylor, etc.) are ranked #1 in their departments and labeled "Above Average"
- Junior employees are at the bottom of their rank and labeled "Below Average"
- Employees whose salary exactly matches the department average (rare, but possible) get "Exactly Average"
- Paula Morris (no department) appears alone — her department average equals her own salary, so she gets "Exactly Average"

**Why this is powerful:** In a non-window-function world, you would need multiple subqueries or CTEs to compute department averages and company averages separately. Window functions let you do it all in one pass over the data.

---

## 12. Try It Yourself Exercises

Now it is your turn. Attempt each exercise before revealing the solution.

---

### Exercise 1: Salary Difference from Department Maximum

Write a query that shows each employee with their name, department, salary, the maximum salary in their department, and the difference between their salary and the department maximum. Order by department and then by the difference ascending (so the person closest to the max salary appears first).

<details>
<summary><strong>Click to reveal solution</strong></summary>

```sql
SELECT 
    e.first_name || ' ' || e.last_name AS full_name,
    d.dept_name,
    e.salary,
    MAX(e.salary) OVER(PARTITION BY e.department_id) AS dept_max_salary,
    MAX(e.salary) OVER(PARTITION BY e.department_id) - e.salary AS gap_to_max
FROM company.employees e
LEFT JOIN company.departments d ON e.department_id = d.dept_id
ORDER BY e.department_id, gap_to_max;
```

**Explanation:** `MAX(salary) OVER(PARTITION BY department_id)` puts the department maximum on every row. Subtracting the employee's own salary gives the gap. The highest earner in each department has a gap of 0.

</details>

In [ ]:
# Exercise 1: Write your solution here
# Hint: Use MAX() OVER(PARTITION BY department_id)

run_query("""
    -- Your query here
    SELECT 1 AS placeholder;
""")

---

### Exercise 2: Running Total of Salary by Hire Date Within Each Department

Write a query that shows each employee ordered by their hire_date within their department, along with a running total of salaries. That is, as you go through employees in hire_date order within a department, accumulate the sum of salaries.

<details>
<summary><strong>Click to reveal solution</strong></summary>

```sql
SELECT 
    e.first_name || ' ' || e.last_name AS full_name,
    d.dept_name,
    e.hire_date,
    e.salary,
    SUM(e.salary) OVER(
        PARTITION BY e.department_id 
        ORDER BY e.hire_date 
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS running_salary_total
FROM company.employees e
LEFT JOIN company.departments d ON e.department_id = d.dept_id
ORDER BY e.department_id, e.hire_date, e.emp_id;
```

**Explanation:** `SUM(salary) OVER(PARTITION BY department_id ORDER BY hire_date)` creates a cumulative sum. The `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` frame makes it explicitly cumulative. Each new hire adds their salary to the running total.

</details>

In [ ]:
# Exercise 2: Write your solution here
# Hint: Use SUM() OVER(PARTITION BY ... ORDER BY ...) with a cumulative frame

run_query("""
    -- Your query here
    SELECT 1 AS placeholder;
""")

---

### Exercise 3: Top Revenue Product Per Region

Write a query that finds the top revenue-generating product in each region. Use a CTE to compute total revenue per (region, product), then use `ROW_NUMBER()` to rank products within each region by revenue, and filter to keep only rank 1. Include the region, product name, total revenue, and rank.

<details>
<summary><strong>Click to reveal solution</strong></summary>

```sql
WITH product_revenue AS (
    SELECT 
        s.region,
        p.product_name,
        ROUND(SUM(s.quantity * p.unit_price), 2) AS total_revenue,
        ROW_NUMBER() OVER(
            PARTITION BY s.region 
            ORDER BY SUM(s.quantity * p.unit_price) DESC
        ) AS rn
    FROM company.sales s
    JOIN company.products p ON s.product_id = p.product_id
    WHERE s.region IS NOT NULL
    GROUP BY s.region, p.product_name
)
SELECT 
    region,
    product_name,
    total_revenue,
    rn AS rank_in_region
FROM product_revenue
WHERE rn = 1
ORDER BY region;
```

**Explanation:** The CTE groups by region and product, computing total revenue. The `ROW_NUMBER() OVER(PARTITION BY region ORDER BY SUM(...) DESC)` ranks products within each region. The outer query filters to `rn = 1` to keep only the top product per region. Note that we can use an aggregate (`SUM(...)`) directly inside the `ORDER BY` of a window function — PostgreSQL handles this elegantly.

</details>

In [ ]:
# Exercise 3: Write your solution here
# Hint: CTE with GROUP BY + ROW_NUMBER() OVER(PARTITION BY region ORDER BY revenue DESC)

run_query("""
    -- Your query here
    SELECT 1 AS placeholder;
""")

---

## Summary

### Window Functions Cheat Sheet

| Function | Purpose | Key Syntax |
|----------|-------|------------|
| `AVG() OVER()` | Aggregate without collapsing rows | `AVG(col) OVER(PARTITION BY x ORDER BY y)` |
| `ROW_NUMBER()` | Unique sequential numbering | `ROW_NUMBER() OVER(PARTITION BY x ORDER BY y)` |
| `RANK()` | Ranking with gaps after ties | `RANK() OVER(PARTITION BY x ORDER BY y)` |
| `DENSE_RANK()` | Ranking without gaps | `DENSE_RANK() OVER(PARTITION BY x ORDER BY y)` |
| `LAG(col, n)` | Access value from n rows before | `LAG(col) OVER(PARTITION BY x ORDER BY y)` |
| `LEAD(col, n)` | Access value from n rows after | `LEAD(col) OVER(PARTITION BY x ORDER BY y)` |
| `SUM() OVER(ORDER BY)` | Running/cumulative total | `SUM(col) OVER(ORDER BY y ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)` |
| `AVG() OVER(ROWS BETWEEN)` | Moving/rolling average | `AVG(col) OVER(ORDER BY y ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)` |
| `PERCENT_RANK()` | Relative position (0 to 1) | `PERCENT_RANK() OVER(PARTITION BY x ORDER BY y)` |
| `NTILE(n)` | Divide into n groups | `NTILE(4) OVER(PARTITION BY x ORDER BY y)` |

### Golden Rules
1. Window functions do **not** collapse rows (unlike `GROUP BY`).
2. Window functions cannot appear in a `WHERE` clause — use a CTE or subquery to filter on their results.
3. `PARTITION BY` divides the data into groups; `ORDER BY` sorts within each group.
4. Adding `ORDER BY` inside `OVER()` changes the default frame from "all rows" to "all rows up to current."
5. Multiple window functions can coexist in the same `SELECT` — each defines its own window.